# Lab 3.2: Content Moderation using Amazon Bedrock Guardrails
## In this notebook

We will learn:
- The content moderation capabilities available in Amazon Bedrock Guardrails
- Practical methods for applying guardrails to standalone text and LLM prompts
- Side-by-side comparison between Amazon Bedrock Guardrails and Amazon Comprehend Trust and Safety

We will complete the following steps in this notebook:
- Create and apply **Protect Wildlife Guardrail** to block harmful content, denied topics and unethical intents in the tour advertisements 
- Create and apply **Ethical Accommodation Guardrail** to ensure that personalized accommodation listings, generated with the help of LLM, are blocked if the original listings suggest violation of Terms & Conditions

----
## [Amazon Bedrock Guardrails](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html)
Amazon Bedrock Guardrails provide configurable safeguards to help safely build generative AI applications at scale. You can create multiple guardrails tailored to different use cases and apply them across LLM prompts or responses or standalone content. 
Amazon Bedrock Guardrails supports the following policies:
- **Content filters** allow to block harmful content categories: Hate, Insults, Sexual, Violence, Misconduct and Prompt Attack.
- **Denied topics** allow to detect and block a set of user-defined topics which are undesirable in the context of a given use case.
- **Word filters** allow to apply filters to block undesirable words, phrases, and profanity (exact match)
- **Sensitive information filters** allow to appl filters to help block or mask sensitive information, such as personally identifiable information (PII), or custom regex in user inputs and model responses. 

For a full list check out the [documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-create.html) 

## Content Moderation on AWS - Service comparison

| Features | Amazon Bedrock <br/> Guardrails | Amazon Comprehend <br/> Trust and Safety |
| --- | --- | --- |
| Modality | Text / Image | Text |
| Evaluated input | LLM Input / Output <br/>Standalone Text / Image | Text segments |
| Toxicity detection | [5 harmful categories](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-content-filters.html) + Custom [denied topics](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-denied-topics.html) | [8 Moderated categories](https://docs.aws.amazon.com/comprehend/latest/dg/trust-safety.html#toxicity-detection) |
| Toxicity dataset | Built-in dataset / Custom phrases,file | Built-in dataset |
| Confidence score | Yes | Yes |
| API | LLM Prompt / Content: [InvokeModel](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_InvokeModel.html) / [Converse](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html) <br/> Standalone Text / Image: [ApplyGuardrail](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_ApplyGuardrail.html) | [DetectToxicContent](https://docs.aws.amazon.com/comprehend/latest/APIReference/API_DetectToxicContent.html) |
| Cost | Per characters | Per characters |
|Additional features | - Profanity filter <br/> - Prompt attacks <br/> - Denied topics | Prompt safety (explicit or implicit malicious intent:<br/> discriminatory, illegal, unsolicited content) |
| PII/PCI handling | Detect by type, regex / Block / Mask | Detect / Redact via [dedicated APIs](https://docs.aws.amazon.com/comprehend/latest/dg/pii.html) |
| Regions | [Supported regions](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-supported.html) | [Supported regions](https://docs.aws.amazon.com/comprehend/latest/dg/trust-safety.html) |

----
## Demo Data Flows
<img src="diagrams/ContentModerationWithBedrockGuardrail.png" alt="Content Moderation With Amazon Bedrock Guardrail">

----
## Pre-requisite steps

Load libraries and update Amazon SageMaker Notebook IAM role with the nessesary permissions

In [ ]:
import boto3
import json
from botocore.config import Config

In [ ]:
iam = boto3.client('iam')
sts = boto3.client('sts')
bedrock = boto3.client('bedrock')
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime',
    config=Config(read_timeout=300)
)
session = boto3.session.Session()

region = session.region_name
account_id = boto3.client('sts').get_caller_identity().get('Account')

In [ ]:
from sagemaker import get_execution_role

policy_name = "CustomGuardrailPolicy"
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:ApplyGuardrail",
                "bedrock:CreateGuardrail",
                "bedrock:CreateGuardrailVersion",
                "bedrock:GetGuardrail",
                "bedrock:DeleteGuardrail"
            ],
            "Resource": [
                f"arn:aws:bedrock:{region}:{account_id}:guardrail/*"
            ]
        },
        
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:ListGuardrails"
            ],
            "Resource": [
                "*"
            ]
        }
    ]
}

# Get current execution role
current_role = get_execution_role()
print(f"Current execution role: {current_role}")

role_name = current_role.split('/')[-1]
response = iam.put_role_policy(
    RoleName=role_name,
    PolicyName=policy_name,
    PolicyDocument=json.dumps(policy_document)
)
print(f"Successfully added inline policy {policy_name} to role {role_name}")

----
## Protect Wildlife Guardrail
### Create a Guardrail

Now lets create a Guardrail to identify harmful content in the travel activities advertisements relying on the following guardrail filters: 
- Content Filters
- Denied Topics
- Word and Phrase Filters 

Run the next lines a) to create guardrail, b) to create guardrail version, c) to review created guardrail version.

This guardrail will be applied to evaluate standalone content from the travel activities advertisements.

In [ ]:
# create a Guardrail to validate text segments from the listings
blocked_msg = """
As a responsible travel service provider, we operate within strict ethical and legal guidelines and can't answer this query.
"""
response = bedrock.create_guardrail(
    name='protect-wildlife-guardrail',
    description='Prevents LLM to unethical travelling recommendations',
    topicPolicyConfig={
        'topicsConfig': [
            {
                'name': 'Prohibted wildlife transportation',
                'definition': 'Tours involving endangered species exploitation and transportation, illegal hunting, or activities harmful to wildlife',
                'examples': [
                    'Recommendations of agencies to expedite international wildlife transportation',
                    'Specialized services to bypass customs clearance'
                ],
                'type': 'DENY'
            }
        ]
    },
    contentPolicyConfig={
        'filtersConfig': [
            {
                'type': 'SEXUAL',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'VIOLENCE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'HATE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'INSULTS',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'MISCONDUCT',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'PROMPT_ATTACK',
                'inputStrength': 'HIGH',
                'outputStrength': 'NONE'
            }
        ]
    },
    wordPolicyConfig={
        'wordsConfig': [
            {'text': 'off the record trips'},
            {'text': 'special arrangements'},
            {'text': 'rare collection'},
            {'text': 'private acquisition'}
        ],
        'managedWordListsConfig': [
            {'type': 'PROFANITY'}
        ]
    },
    contextualGroundingPolicyConfig={
        'filtersConfig': [
            {
                'type': 'GROUNDING',
                'threshold': 0.9
            },
            {
                'type': 'RELEVANCE',
                'threshold': 0.9
            }
        ]
    },
    blockedInputMessaging=blocked_msg,
    blockedOutputsMessaging=blocked_msg,
)
guardrail_id = response['guardrailId']
print(f"The guardrail {guardrail_id} has been created")

In [ ]:
# create a version for the Guardrail
response = bedrock.create_guardrail_version(
    guardrailIdentifier=guardrail_id,
    description='Guardrail Version'
)
guardrail_version = '1'

In [ ]:
# review a version for the Guardrail
response = bedrock.get_guardrail(
    guardrailIdentifier=guardrail_id,
    guardrailVersion='1'
)
print(response)
guardrail_id_protect_wildlife = guardrail_id

### Apply Guardrail 

Next, we apply our guardrail to analyze a travel activity, advertising rare bird viewing. The applied guardrail can help identify concerning content that suggests suspicious wildlife acquisition and illegal transportation methods.

In [ ]:
# apply our Guardrail to the listing advertising tour to see rare birds
prompt = """
Discover the world's most extraordinary bird collections:
Travel with us to exclusive destinations showcasing the planet's most exclusive and rare bird species. 
Our tours with special arrangements connect discerning collectors with unique acquisition opportunities not available to the general public.

Premier wildlife transit solutions:
Our network of expert logistics partners specializes in handling international wildlife transportation bypassing customs clearance
"""

response = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=guardrail_id,
    guardrailVersion=guardrail_version,
    source='INPUT',
    content=[
        {
            "text": {
                "text": prompt
            }
        }
    ]
)
print("Blocked content:")
print(json.dumps(response['assessments'], indent=2))

## Ethical Accommodation Guardrail
### Create a Guardrail

Now let's create a guardrail that blocks the personalization of accommodation listings containing toxic content or harmful intent that may violate Terms and Conditions, ensuring the accommodation platform operates within legal and ethical boundaries. This guardrail is also designed to block personally identifiable information (PII) in the original listing to prevent unauthorized direct communication between hosts and travelers outside the platform.

This guardrail will be integrated into the LLM workflow used for personalizing accommodation listings.

In [ ]:
# create a Guardrail to validate LLM content
blocked_msg = """
As a responsible accommodation service provider, we operate within strict ethical and legal guidelines and can't post this listing.
"""
response = bedrock.create_guardrail(
    name='ethical-accommodation-guardrail',
    description='Prevents LLM to produce listings in violation of Terms & Conditions',
    topicPolicyConfig={
        'topicsConfig': [
            {
                'name': 'Terms and Conditions Violations',
                'definition': 'Instances that violate Terms & Conditions or ethical boundaries in accommodation listings',
                'examples': [
                    'Avoid mentioning about your stay to building management or neighbors',
                    'Avoid listing fees by choosing cash payments upon arrival'
                ],
                'type': 'DENY'
            }
        ]
    },
    contentPolicyConfig={
        'filtersConfig': [
            {
                'type': 'SEXUAL',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'VIOLENCE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'HATE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'INSULTS',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'MISCONDUCT',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'PROMPT_ATTACK',
                'inputStrength': 'HIGH',
                'outputStrength': 'NONE'
            }
        ]
    },
    wordPolicyConfig={
        'wordsConfig': [
            {'text': 'off the books'},
            {'text': 'avoid fees'},
            {'text': 'cash only'},
            {'text': 'based on ethnicity'}
        ],
        'managedWordListsConfig': [
            {'type': 'PROFANITY'}
        ]
    },
    sensitiveInformationPolicyConfig={
        'piiEntitiesConfig': [
            {'type': 'EMAIL', 'action': 'BLOCK'},
            {'type': 'PHONE', 'action': 'BLOCK'}
        ]
    },
    contextualGroundingPolicyConfig={
        'filtersConfig': [
            {
                'type': 'GROUNDING',
                'threshold': 0.9
            },
            {
                'type': 'RELEVANCE',
                'threshold': 0.9
            }
        ]
    },
    blockedInputMessaging=blocked_msg,
    blockedOutputsMessaging=blocked_msg,
)
guardrail_id = response['guardrailId']
print(f"The guardrail {guardrail_id} has been created")

In [ ]:
# create a version for the Guardrail
response = bedrock.create_guardrail_version(
    guardrailIdentifier=guardrail_id,
    description='Guardrail Version'
)
guardrail_version = '1'
guardrail_id_ethical_listings = guardrail_id

### Apply Guardrail 

Next, we implement our guardrail when generating a personalized listing for our fictional traveler based on the host's original listing. The guardrail serves two critical functions: it flags potentially problematic content that might violate Terms and Conditions, while also blocking personal information within the accommodation listing. This prevents unauthorized direct communications between hosts and travellers outside the accommodation platform.

Before running the next cell, enable model access for Claude 3 Haiku LLM:
1. Go to Amazon Bedrock https://console.aws.amazon.com/bedrock/home#/modelaccess
2. Modify model access
3. Enable model access for *Anthropic - Claude 3 Haiku*
4. Save changes

In [ ]:
# Build our request to Bedrock, we will test our guardrail version
model_id = 'anthropic.claude-3-haiku-20240307-v1:0'
prompt = """
Personalize the following listing to a potential customer:
<listing>
Cozy Downtown Loft - Perfect Urban Retreat
Enjoy our stylish loft in the heart of downtown! 
This 2-bedroom space features modern amenities, a fully equipped kitchen, and stunning city views.

House Rules:
No parties or events
No smoking inside the property
We reserve the right to decline guests based on ethnicity
Please pay cleaning fee in cash to avoid transaction records
Do not mention to neighbors you are a short-term guest

Call us on +61478320943 to secure a booking without any fees.
</listing>
"""

payload = {
    "modelId": model_id,
    "contentType": "application/json",
    "accept": "application/json",
    "body": {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ]
    }
}

# Convert the payload to bytes
body_bytes = json.dumps(payload['body']).encode('utf-8')

# Invoke the model
response = bedrock_runtime.invoke_model(
    body = body_bytes,
    contentType = payload['contentType'],
    accept = payload['accept'],
    modelId = model_id,
    guardrailIdentifier = guardrail_id, 
    guardrailVersion =guardrail_version, 
    trace = "ENABLED"
)

# Print the response
response_body = response['body'].read().decode('utf-8')
print(json.dumps(json.loads(response_body), indent=2))

# Summary

In this notebook, we demonstrated how to leverage [Amazon Bedrock Guardrails](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails.html) to identify unwanted content across supported [5 harmful categories](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-content-filters.html), as well as content containing custom [denied topics](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-denied-topics.html), [phrases](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-word-filters.html), and [sensitive details](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-sensitive-filters.html). We learned how to apply guardrails to LLM inputs and outputs by using inference APIs [InvokeModel](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_InvokeModel.html) or [Converse](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_Converse.html), and how to use guardrails beyond LLM applications for standalone text and images by using API [ApplyGuardrail](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_ApplyGuardrail.html).

# Cleanup

To clean up the Amazon Bedrock Guardrails resources created in this notebook, you can execute the section **Clean up Amazon Bedrock Guardrails** in the [`04_Clean_Up.ipynb`](04_Clean_Up.ipynb) notebook. If you're running these notebooks as part of an AWS-led workshop where temporary AWS accounts are provided for you, this cleanup will be done automatically for you. Otherwise, if you're running this notebook in a personal or work account, be sure to run the [`04_Clean_Up.ipynb`](04_Clean_Up.ipynb) notebook to shutdown resources that can create ongoing AWS charges.